In [2]:
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu pypdf fastapi uvicorn python-multipart pyngrok nest_asyncio

## 1) Load the quantized model (4-bit)

In [3]:
import torch, json, re, io, threading
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_PATH = None   
API_KEY = "shahd123" 

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
if ADAPTER_PATH:
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model.eval()
print(f"GPU 0 memory: {torch.cuda.memory_allocated(0)/1e9:.1f} GB")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

GPU 0 memory: 2.5 GB


In [4]:
gpu_lock = threading.Lock()

def chat(system, user, max_new_tokens=700, temperature=0.0):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    gen = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id)
    if temperature > 0:
        gen.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        gen.update(do_sample=False)
    with gpu_lock, torch.no_grad():
        out = model.generate(**inputs, **gen)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(chat("You are a helpful assistant.", "Say hello in one sentence."))

Hello!


## 2) Structured extraction (Pydantic output parser + retry)

In [5]:
from pydantic import BaseModel, ValidationError

class PaperInfo(BaseModel):
    title: str
    problem: str
    method: str
    dataset: str
    results: str
    limitations: str

EXTRACT_SYS = (
    "You are an expert research assistant. Read the paper text and return ONLY a valid JSON object "
    "with these string fields: title, problem, method, dataset, results, limitations. "
    "Use short, factual sentences. If something is not stated write \"not stated\". "
    "No markdown, no extra text."
)

def parse_json(raw):
    start, end = raw.find("{"), raw.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("no JSON object found")
    return json.loads(raw[start:end + 1])

def extract_info(paper_text, retries=2):
    if len(paper_text) > 18000:
        paper_text = paper_text[:14000] + "\n...\n" + paper_text[-4000:]
    last_err = ""
    for _ in range(retries + 1):
        prompt = f"Paper text:\n{paper_text}"
        if last_err:
            prompt += f"\n\nYour previous answer was invalid ({last_err}). Return valid JSON only."
        raw = chat(EXTRACT_SYS, prompt, max_new_tokens=600)
        try:
            return PaperInfo.model_validate(parse_json(raw)).model_dump()
        except (ValueError, ValidationError) as e:
            last_err = str(e)[:200]
    return None

## 3) RAG with section-aware chunking

In [6]:
import numpy as np, faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-base-en-v1.5", device="cuda")
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

HEAD = re.compile(
    r"^\s*(?:\d+\.?\s*)?(abstract|introduction|related work|background|methodology|methods?|"
    r"experiments?|results|discussion|conclusions?)\s*$", re.I | re.M)

def split_sections(text):
    marks = [(m.start(), m.group(1).title()) for m in HEAD.finditer(text)]
    if not marks:
        return [("Full text", text)]
    sections = []
    if marks[0][0] > 0:
        sections.append(("Front matter", text[:marks[0][0]]))
    for i, (pos, name) in enumerate(marks):
        end = marks[i + 1][0] if i + 1 < len(marks) else len(text)
        sections.append((name, text[pos:end]))
    return sections

def chunk_words(text, size=200, overlap=40):
    words = text.split()
    out = []
    for i in range(0, len(words), size - overlap):
        piece = words[i:i + size]
        if piece:
            out.append(" ".join(piece))
    return out

papers = {}   # name -> {"text": str, "info": dict | None}
chunks = []   # {"paper", "section", "text"}
index = None

def rebuild_index():
    global index
    chunks.clear()
    for name, p in papers.items():
        for sec, body in split_sections(p["text"]):
            for c in chunk_words(body):
                chunks.append({"paper": name, "section": sec, "text": c})
    if not chunks:
        index = None
        return
    emb = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True,
                          convert_to_numpy=True, batch_size=32).astype("float32")
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)

def retrieve(question, k=5):
    q = embedder.encode([QUERY_PREFIX + question], normalize_embeddings=True,
                        convert_to_numpy=True).astype("float32")
    _, idx = index.search(q, min(k, len(chunks)))
    return [chunks[i] for i in idx[0]]

ANSWER_SYS = ("Answer the question using ONLY the context passages. Cite sources as [paper | section]. "
              "If the answer is not in the context, say you cannot find it.")

def answer_question(question, k=5):
    ctx = retrieve(question, k)
    context = "\n\n".join(f"[{c['paper']} | {c['section']}]\n{c['text']}" for c in ctx)
    ans = chat(ANSWER_SYS, f"Context:\n{context}\n\nQuestion: {question}", max_new_tokens=500)
    return ans, ctx

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 4) Chains: comparison table -> Related Work, and summarizer (LoRA style)

In [8]:
COMPARE_SYS = ("You are an academic writer. Given structured summaries of several papers, produce a markdown "
               "table with one row per paper and columns: Paper, Problem, Method, Dataset, Key result, "
               "Limitation. Output only the table.")
RELATED_SYS = ("You write the 'Related Work' section of a research paper. Using the comparison table, write "
               "2 coherent academic paragraphs that group the papers by approach, point out similarities "
               "and gaps, and refer to papers by title. Do not invent facts that are not in the table.")
SUM_SYS = (
    "Write an academic-style abstract (120-180 words) summarizing this paper. "
    "Structure: 1) the problem, 2) the proposed method, 3) the data and key results, 4) a short conclusion. "
    "Use formal, third-person, past-tense scientific language. Do not use bullet points or invent numbers."
)

def compare_papers():
    infos = {n: p["info"] for n, p in papers.items() if p["info"]}
    table = chat(COMPARE_SYS, json.dumps(infos, indent=1), max_new_tokens=900)
    related = chat(RELATED_SYS, table, max_new_tokens=700, temperature=0.3)
    return table, related

def summarize(name):
    return chat(SUM_SYS, papers[name]["text"][:14000], max_new_tokens=350)

## 5) FastAPI

In [9]:
from fastapi import FastAPI, UploadFile, File, Depends, HTTPException, Header
from pypdf import PdfReader

app = FastAPI(title="Literature Agent")

def auth(authorization: str = Header(default="")):
    if authorization != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Invalid API key")

class AskReq(BaseModel):
    question: str
    k: int = 5

class NameReq(BaseModel):
    name: str

@app.get("/health")
def health():
    return {"ok": True, "papers": len(papers)}

@app.post("/upload", dependencies=[Depends(auth)])
def upload(file: UploadFile = File(...)):
    reader = PdfReader(io.BytesIO(file.file.read()))
    text = "\n".join((pg.extract_text() or "") for pg in reader.pages)
    if len(text.split()) < 200:
        raise HTTPException(400, "Could not read text from this PDF (scanned?)")
    papers[file.filename] = {"text": text, "info": extract_info(text)}
    rebuild_index()
    return {"name": file.filename, "info": papers[file.filename]["info"]}

@app.get("/papers", dependencies=[Depends(auth)])
def list_papers():
    return [{"name": n, "info": p["info"]} for n, p in papers.items()]

@app.post("/ask", dependencies=[Depends(auth)])
def ask(req: AskReq):
    if index is None:
        raise HTTPException(400, "Upload papers first")
    ans, ctx = answer_question(req.question, req.k)
    return {"answer": ans, "sources": ctx}

@app.post("/compare", dependencies=[Depends(auth)])
def compare():
    if len([p for p in papers.values() if p["info"]]) < 2:
        raise HTTPException(400, "Upload at least 2 papers")
    table, related = compare_papers()
    return {"table": table, "related_work": related}

@app.post("/summarize", dependencies=[Depends(auth)])
def summarize_ep(req: NameReq):
    if req.name not in papers:
        raise HTTPException(404, "Unknown paper")
    return {"summary": summarize(req.name)}

@app.post("/reset", dependencies=[Depends(auth)])
def reset():
    papers.clear()
    rebuild_index()
    return {"ok": True}

## 6) Expose with ngrok and run (this cell keeps running)

In [ ]:
import uvicorn
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

ngrok.kill()   
ngrok.set_auth_token(UserSecretsClient().get_secret("NGROK_TOKEN"))
public_url = ngrok.connect(8000).public_url
print("Public URL:", public_url)

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

Public URL: https://reclusive-curse-zodiac.ngrok-free.dev


INFO:     Started server process [151]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     156.201.105.153:0 - "GET /papers HTTP/1.1" 200 OK


/usr/local/lib/python3.13/dist-packages/pypdf/generic/_data_structures.py:1585: RuntimeWarning: coroutine 'Server.serve' was never awaited
  peek = stream.read(20)


INFO:     156.201.105.153:0 - "POST /upload HTTP/1.1" 200 OK
INFO:     156.201.105.153:0 - "GET /papers HTTP/1.1" 200 OK
INFO:     156.201.105.153:0 - "GET /papers HTTP/1.1" 200 OK
INFO:     156.201.105.153:0 - "GET /papers HTTP/1.1" 200 OK
INFO:     156.201.105.153:0 - "GET /papers HTTP/1.1" 200 OK
INFO:     156.201.105.153:0 - "POST /summarize HTTP/1.1" 200 OK
INFO:     156.201.105.153:0 - "POST /ask HTTP/1.1" 200 OK
INFO:     156.201.105.153:0 - "GET /papers HTTP/1.1" 200 OK
INFO:     156.201.105.153:0 - "GET /papers HTTP/1.1" 200 OK
INFO:     156.201.105.153:0 - "GET /papers HTTP/1.1" 200 OK
INFO:     156.201.105.153:0 - "GET /papers HTTP/1.1" 200 OK
INFO:     156.201.105.153:0 - "POST /upload HTTP/1.1" 200 OK
INFO:     156.201.105.153:0 - "POST /upload HTTP/1.1" 200 OK
INFO:     156.201.105.153:0 - "GET /papers HTTP/1.1" 200 OK
INFO:     156.201.105.153:0 - "GET /papers HTTP/1.1" 200 OK
INFO:     156.201.105.153:0 - "GET /papers HTTP/1.1" 200 OK
INFO:     156.201.105.153:0 - "GET 

In [ ]:
 https://reclusive-curse-zodiac.ngrok-free.dev